# DITTO training on Colab: tuning on dev, 3 seeds, one test evaluation

Runs top to bottom with **Runtime > Run all**. Pick **A100** (best) or **L4** under *Runtime > Change runtime
type*. T4 works but runs in float32, much slower, and skips the 4B variant.

What it does:
1. Checks the GPU and picks the dtype (bfloat16 on A100/L4, float32 on T4).
2. Mounts Google Drive. Every run, evaluation and table is written to `MyDrive/aphasia_ditto_colab_v2/` as
   it goes (a new folder: nothing from the earlier small-split runs is reused). After a disconnect, run all
   cells again: finished runs and evaluations are reused.
3. Clones `feature/ditto-training`, installs it with `pip install -e ".[llm]"`, removes the preinstalled
   torchao (incompatible with peft) and checks that a LoRA adapter can be created.
4. **Tuning on the new dev split only** (33 situations, 3 per rule variant; test labels are never read
   here). Five variants, each changing one thing from the current config, all trained on the same 119
   training situations: current config, more SFT steps, more DPO steps per round, option scoring instead
   of free generation, and Qwen3-4B instead of 1.7B (if GPU memory allows).
5. Picks the best variant by dev agreement (ties: smaller model, then faster) and saves the choice.
6. Trains seeds 1 and 2 of that variant (seed 0 is its tuning run). The delivered seed is chosen on dev
   and saved, before any test evaluation.
7. Evaluates **once** on the new test split (55 situations, 5 per rule variant, pictures never the current
   picture in training) and on the old 10 test situations (`test_v1`): base, prompted (20 v1 demonstrations
   in the prompt), prompted_all (every training demonstration that fits the context window: all 119),
   SFT and DITTO. Agreement with 95% bootstrap intervals, mean and range across seeds, paired differences
   DITTO minus each baseline, valid-output rate, latency and per-rule accuracy.
8. Zips everything into one file in Drive and offers the best adapter and tables for download.

The demonstrations are fictional, generated from demonstrator A's written strategy and pending human
review.

In [ ]:
# 1. GPU check, dtype and rough time estimates
import subprocess, sys, time
import torch

assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > GPU (A100 or L4), then run all again."
GPU = torch.cuda.get_device_name(0)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
major, _ = torch.cuda.get_device_capability(0)
if major >= 8 and torch.cuda.is_bf16_supported():
    DTYPE = "bfloat16"
else:
    # No native bfloat16 (T4 and older). float32 rather than float16: LoRA training in pure float16
    # with AdamW can overflow or underflow; float32 is safe for a 1.7B model in 16 GB.
    DTYPE = "float32"
    print("WARNING: this GPU has no native bfloat16. Using float32: safe but about 3-5x slower than an L4, "
          "and the 4B variant is skipped. Prefer an A100 or L4 if available.")
ALLOW_4B = DTYPE == "bfloat16" and GPU_GB >= 20
SPEED = 1.0 if "A100" in GPU or "H100" in GPU else 0.5 if "L4" in GPU else 0.2 if "T4" in GPU else 0.4
print(f"GPU: {GPU}, {GPU_GB:.0f} GB, dtype {DTYPE}, Qwen3-4B variant: {'yes' if ALLOW_4B else 'skipped'}")

# Rough minutes on an A100 (not measured on Colab; the notebook prints real times as it goes).
A100_MIN = {"setup (clone, install, model download)": 6,
            "tuning: 1.7B variants (4 runs + dev eval)": 30,
            "tuning: 4B variant (1 run + dev eval)": 16 if ALLOW_4B else 0,
            "seeds 1 and 2 of the chosen variant (+ dev eval)": 16,
            "test + test_v1 evaluation (once; prompted_all has ~23k-token prompts)": 12, "zip": 2}
for stage, m in A100_MIN.items():
    print(f"  {stage:70s} ~{m / SPEED:4.0f} min")
print(f"  {'total':70s} ~{sum(A100_MIN.values()) / SPEED:4.0f} min")

In [ ]:
# 2. Google Drive: everything persistent lives here
from google.colab import drive
drive.mount("/content/drive")
import json, os, shutil
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive/aphasia_ditto_colab_v2")
RUNS, EVAL, CONFIGS, LOGS, RESULTS = (DRIVE / d for d in ("runs", "eval", "configs", "logs", "results"))
for d in (RUNS, EVAL, CONFIGS, LOGS, RESULTS):
    d.mkdir(parents=True, exist_ok=True)
print("saving to", DRIVE)
print("already there:", sorted(p.name for p in RUNS.iterdir()) or "nothing (fresh start)")

In [ ]:
# 3. Clone the branch and install
REPO_URL, BRANCH, REPO = "https://github.com/burakdon/HackathonPOPO.git", "feature/ditto-training", Path("/content/HackathonPOPO")
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "fetch", "-q", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "-q", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "-q", "--ff-only"], check=True)
os.chdir(REPO)
# An editable install is only picked up by a fresh kernel; putting the repo on sys.path makes its
# modules importable in this session too.
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("commit", COMMIT)
# The project pins Python 3.11 for uv; Colab runs 3.12, which the code supports, hence --ignore-requires-python.
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--ignore-requires-python", "-e", ".[llm]"],
                   capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-1500:])
assert r.returncode == 0, "pip install failed (see above)"
# Colab preinstalls torchao 0.10, which peft rejects ("only versions above 0.16.0 are supported");
# training does not use it, so remove it before peft is first imported.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
LORA_CHECK = '''
import peft, torch
from peft import LoraConfig, get_peft_model
from transformers import Qwen3Config, Qwen3ForCausalLM
m = Qwen3ForCausalLM(Qwen3Config(vocab_size=128, hidden_size=32, intermediate_size=64, num_hidden_layers=1,
                                 num_attention_heads=2, num_key_value_heads=1, head_dim=16))
m = get_peft_model(m, LoraConfig(r=4, lora_alpha=8, target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM"))
loss = m(input_ids=torch.tensor([[1, 2, 3]]), labels=torch.tensor([[1, 2, 3]])).loss
loss.backward()
print("LORA_OK peft", peft.__version__, "trainable", sum(p.numel() for p in m.parameters() if p.requires_grad))
'''
chk = subprocess.run([sys.executable, "-c", LORA_CHECK], capture_output=True, text=True)
print(chk.stdout.strip(), chk.stderr[-2000:] if chk.returncode else "")
assert "LORA_OK" in chk.stdout, "peft / LoRA self-check failed (see above); do not start training"
import transformers, peft, accelerate
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__,
      "| accelerate", accelerate.__version__)

info_path = DRIVE / "run_info.json"
info = json.loads(info_path.read_text()) if info_path.exists() else {"sessions": []}
if info["sessions"] and info["sessions"][0]["commit"] != COMMIT:
    print(f"WARNING: earlier runs in Drive used commit {info['sessions'][0]['commit'][:8]}, now {COMMIT[:8]}.")
info["sessions"].append({"commit": COMMIT, "gpu": GPU, "dtype": DTYPE, "started": time.strftime("%Y-%m-%d %H:%M:%S"),
                         "torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__})
info_path.write_text(json.dumps(info, indent=1))

In [ ]:
# 3b. Hugging Face login only if needed (the Qwen3 models are public)
from huggingface_hub import model_info

BASE_17 = ("Qwen/Qwen3-1.7B", "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e")
BASE_4 = ("Qwen/Qwen3-4B", "1cfa9a7208912126459214e8b04321603b3df60c")
needed = [BASE_17] + ([BASE_4] if ALLOW_4B else [])
try:
    for m, rev in needed:
        model_info(m, revision=rev)
    print("models reachable without login:", [m for m, _ in needed])
except Exception as e:
    print("model not reachable anonymously:", type(e).__name__, "- logging in")
    from huggingface_hub import login
    try:
        from google.colab import userdata
        login(userdata.get("HF_TOKEN"))          # Colab secret named HF_TOKEN
    except Exception:
        from huggingface_hub import notebook_login
        notebook_login()

## Helpers: configs, resumable training and evaluation
Each variant gets its own practice config (base model, revision, dtype, decoding, Drive directories)
and DITTO config (repo `configs/ditto_llm.yaml` plus the variant's changes, v2 demonstrations). A run
counts as finished only if its manifest says `completed`; partial runs are deleted and redone. An
evaluation is reused if its JSON exists in Drive.

In [ ]:
import yaml

# v3: the same 119 training situations as v2, plus the new dev (33) / test (55) splits and test_v1 (10).
DEMOS = "data/demonstrations/demonstrator_a_v3.jsonl"
VARIANTS = {
    "v1_current":  {"base": BASE_17, "decoding": "generate", "changes": {},
                    "what": "configs/ditto_llm.yaml as is (SFT early stop at loss 0.15, 3 rounds x 15 DPO steps)"},
    "v2_more_sft": {"base": BASE_17, "decoding": "generate", "changes": {"sft.max_steps": 120, "sft.stop_loss": 0.0},
                    "what": "120 SFT steps, no early stop (about 4 epochs of 119 situations)"},
    "v3_more_dpo": {"base": BASE_17, "decoding": "generate", "changes": {"ditto.steps_per_round": 45},
                    "what": "45 DPO steps per round instead of 15"},
    "v4_score":    {"base": BASE_17, "decoding": "score", "changes": {},
                    "what": "choose by scoring each valid option's log-probability (inference and DITTO sampling)"},
}
if ALLOW_4B:
    VARIANTS["v5_qwen3_4b"] = {"base": BASE_4, "decoding": "generate", "changes": {},
                               "what": "Qwen3-4B instead of Qwen3-1.7B"}
SIZE_B = {BASE_17[0]: 1.7, BASE_4[0]: 4.0}


def practice_path(variant: str) -> Path:
    v = VARIANTS[variant]
    p = yaml.safe_load((REPO / "configs/practice.yaml").read_text())
    p["llm"].update(base_model=v["base"][0], revision=v["base"][1], dtype=DTYPE, decoding=v["decoding"],
                    runs_dir=str(RUNS), timeout_s=120)
    p["eval_dir"] = str(EVAL)
    out = CONFIGS / f"practice_{variant}.yaml"
    out.write_text(yaml.safe_dump(p, sort_keys=False))
    return out


def ditto_path(variant: str, seed: int) -> Path:
    d = yaml.safe_load((REPO / "configs/ditto_llm.yaml").read_text())
    d.update(practice_config=str(practice_path(variant)), demonstrations=DEMOS, seed=seed)
    for key, val in VARIANTS[variant]["changes"].items():
        section, name = key.split(".")
        d[section][name] = val
    out = CONFIGS / f"ditto_{variant}_s{seed}.yaml"
    out.write_text(yaml.safe_dump(d, sort_keys=False))
    return out


def run_id(variant: str, seed: int) -> str:
    return f"{variant}-s{seed}"


def stream(cmd: list[str], log: Path) -> None:
    # Run a module, echo its output (minus progress bars) and keep a full log in Drive.
    with open(log, "a") as f, subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                               text=True, bufsize=1) as p:
        for line in p.stdout:
            f.write(line)
            if "it/s]" not in line and "s/it]" not in line and "Warning" not in line:
                print(line, end="")
    if p.returncode != 0:
        raise RuntimeError(f"command failed ({p.returncode}); log: {log}")


def manifest(rid: str) -> dict | None:
    p = RUNS / rid / "manifest.json"
    return json.loads(p.read_text()) if p.exists() else None


def train(variant: str, seed: int) -> dict:
    rid = run_id(variant, seed)
    m = manifest(rid)
    if m and m.get("status") == "completed":
        print(f"[skip] {rid} already trained ({m['seconds']} s)")
        return m
    if (RUNS / rid).exists():
        print(f"[redo] {rid} was incomplete (status {m and m.get('status')}); deleting it")
        shutil.rmtree(RUNS / rid)
    print(f"[train] {rid}: {VARIANTS[variant]['what']}")
    t0 = time.time()
    stream([sys.executable, "-m", "training.ditto", "--config", str(ditto_path(variant, seed)), "--run-id", rid],
           LOGS / f"train_{rid}.log")
    print(f"[done] {rid} in {(time.time() - t0) / 60:.1f} min")
    return manifest(rid)


def evaluate(variant: str, seed: int, split: str, conditions: list[str]) -> dict:
    rid = run_id(variant, seed)
    out = EVAL / f"{rid}-{split}.json"
    if out.exists():
        print(f"[skip] {split} evaluation of {rid} already in Drive")
        return json.loads(out.read_text())
    conds = [c.format(rid=rid) for c in conditions]
    stream([sys.executable, "-m", "evaluation.compare", "--run", rid, "--split", split,
            "--practice", str(practice_path(variant)), "--conditions", *conds], LOGS / f"eval_{rid}_{split}.log")
    return json.loads(out.read_text())


STAGE_TIMES = {}

## 4. Tuning: dev split only
Each variant: train with seed 0, then evaluate base, SFT and DITTO on the 33 dev situations.
`evaluation.compare --split dev` reads only dev rows; nothing in this section touches test.

In [ ]:
t_stage = time.time()
dev = {}
for variant in VARIANTS:
    m = train(variant, 0)
    rep = evaluate(variant, 0, "dev", ["base", "sft:{rid}", "ditto:{rid}"])
    dev[variant] = {"manifest": m, "report": rep}
STAGE_TIMES["tuning"] = time.time() - t_stage


from evaluation.compare import bootstrap_ci, fmt_agree, to_markdown


def row(variant: str) -> dict:
    m, res = dev[variant]["manifest"], dev[variant]["report"]["results"]
    rid = run_id(variant, 0)
    d, s, b = res[f"ditto:{rid}"], res[f"sft:{rid}"], res["base"]
    return {"variant": variant, "base model": m["base_model"].split("/")[1], "decoding": m["decoding"],
            "dev agree DITTO": d["agree_end_to_end"], "dev agree SFT": s["agree_end_to_end"],
            "dev agree base": b["agree_end_to_end"], "n": d["n"], "DITTO valid": d["valid_output"],
            "DITTO median latency ms": d["latency_ms_median"], "train min": round(m["seconds"] / 60, 1),
            "SFT steps": m["sft"]["steps"], "what": VARIANTS[variant]["what"],
            "DITTO (95% CI)": fmt_agree(d["agree_end_to_end"], d["n"], d["agree_end_to_end_ci95"]),
            "SFT (95% CI)": fmt_agree(s["agree_end_to_end"], s["n"], s["agree_end_to_end_ci95"]),
            "base (95% CI)": fmt_agree(b["agree_end_to_end"], b["n"], b["agree_end_to_end_ci95"])}


import pandas as pd
tuning = pd.DataFrame([row(v) for v in VARIANTS])
tuning.to_csv(RESULTS / "tuning_dev.csv", index=False)
print(f"tuning took {STAGE_TIMES['tuning'] / 60:.1f} min")
tuning[["variant", "base model", "decoding", "DITTO (95% CI)", "SFT (95% CI)", "base (95% CI)",
        "DITTO median latency ms", "train min"]]

## 5. Choose the variant (dev agreement; ties: smaller model, then lower latency)
The choice is saved to Drive and reused on resume, so it cannot change after the test evaluation.

In [ ]:
choice_path = DRIVE / "choice.json"
if choice_path.exists():
    choice = json.loads(choice_path.read_text())
    print("[reuse] choice made earlier:", choice["variant"])
else:
    ranked = sorted(VARIANTS, key=lambda v: (-row(v)["dev agree DITTO"], SIZE_B[VARIANTS[v]["base"][0]],
                                             row(v)["DITTO median latency ms"]))
    best = ranked[0]
    r = row(best)
    tied = [v for v in ranked[1:] if row(v)["dev agree DITTO"] == r["dev agree DITTO"]]
    why = (f"{best} has the highest DITTO agreement on dev ({r['dev agree DITTO']}/{r['n']})"
           + (f"; tied with {', '.join(tied)}, broken by smaller model then lower latency "
              f"({r['base model']}, {r['DITTO median latency ms']:.0f} ms)" if tied else ""))
    choice = {"variant": best, "why": why, "ranking": ranked, "dev_table": tuning.to_dict("records")}
    choice_path.write_text(json.dumps(choice, indent=1))
CHOSEN = choice["variant"]
print("CHOSEN:", CHOSEN, "-", VARIANTS[CHOSEN]["what"])
print("WHY:", choice["why"])

## 6. Seeds 1 and 2 of the chosen variant
Seed 0 is the tuning run. The seed delivered as the adapter is chosen on **dev** (ties: lowest seed),
never on test.

In [ ]:
t_stage = time.time()
SEEDS = [0, 1, 2]
seed_dev = {}
for seed in SEEDS:
    train(CHOSEN, seed)
    rep = evaluate(CHOSEN, seed, "dev", ["base", "sft:{rid}", "ditto:{rid}"])
    seed_dev[seed] = rep["results"][f"ditto:{run_id(CHOSEN, seed)}"]["agree_end_to_end"]
STAGE_TIMES["seeds"] = time.time() - t_stage
BEST_SEED = sorted(SEEDS, key=lambda s: (-seed_dev[s], s))[0]
print("dev agreement per seed:", seed_dev, "-> delivered seed:", BEST_SEED)
json.dump({"seed_dev": seed_dev, "best_seed": BEST_SEED}, open(DRIVE / "seed_choice.json", "w"))

## 7. Test evaluation: exactly once (new test split and test_v1)
The variant and the delivered seed are already saved. Seed 0's run is evaluated with every condition
(base, prompted, prompted_all and bandit do not depend on the seed), seeds 1 and 2 with SFT and DITTO.
Results are stored in Drive; a rerun reloads them instead of evaluating again. Intervals: 95% percentile
bootstrap over situations; for SFT and DITTO each situation's agreement is first averaged over the 3 seeds.

In [ ]:
import numpy as np
from statistics import mean

assert (DRIVE / "choice.json").exists() and (DRIVE / "seed_choice.json").exists(), "choose variant and seed first"
TEST_SPLITS = ["test", "test_v1"]
SEED0 = ["base", "prompted", "prompted_all", "sft:{rid}", "ditto:{rid}", "bandit"]
t_stage = time.time()
reports = {}
for split in TEST_SPLITS:
    reports[split] = {0: evaluate(CHOSEN, 0, split, SEED0)}
    for seed in SEEDS[1:]:
        reports[split][seed] = evaluate(CHOSEN, seed, split, ["sft:{rid}", "ditto:{rid}"])
STAGE_TIMES["test"] = time.time() - t_stage

GROUPS = {"base": ["base"], "prompted": ["prompted"], "prompted_all": ["prompted_all"],
          "SFT": [f"sft:{run_id(CHOSEN, s)}" for s in SEEDS], "DITTO": [f"ditto:{run_id(CHOSEN, s)}" for s in SEEDS],
          "bandit": ["bandit"]}


def analyse(split: str) -> tuple[list[str], dict]:
    test = reports[split]
    assert all(test[s]["example_ids"] == test[0]["example_ids"] for s in SEEDS)
    res = {"base": test[0]["results"]["base"], "prompted": test[0]["results"]["prompted"],
           "prompted_all": test[0]["results"]["prompted_all"]}
    for kind in ("sft", "ditto"):
        for seed in SEEDS:
            c = f"{kind}:{run_id(CHOSEN, seed)}"
            res[c] = test[seed]["results"][c]
    res["bandit"] = test[0]["results"]["bandit"]
    combined = {**test[0], "run_id": f"{CHOSEN} (seeds {', '.join(map(str, SEEDS))})", "results": res}
    n = res["base"]["n"]
    # per-situation agreement, averaged over seeds within a group
    hits = {g: np.mean([[p["agree_end_to_end"] for p in res[c]["per_example"]] for c in conds], axis=0)
            for g, conds in GROUPS.items()}
    summary = []
    for g, conds in GROUPS.items():
        ag = [res[c]["agree_end_to_end"] for c in conds]
        va = [res[c]["valid_output"] for c in conds if res[c]["valid_output"] is not None]
        ci = bootstrap_ci(hits[g])
        shots = res[conds[0]].get("model_detail", {}).get("few_shot_examples")
        summary.append({"model": g + (f" ({shots} demos)" if g.startswith("prompted") and shots else ""),
                        "runs": len(conds), "agree": fmt_agree(round(mean(ag), 2), n, ci),
                        "range over seeds": f"{min(ag)}-{max(ag)}" if len(conds) > 1 else "",
                        "valid output": f"{mean(va):.1f}/{n}" if va else "n/a",
                        "median latency ms": round(mean(res[c]["latency_ms_median"] for c in conds))})
    # paired differences DITTO - baseline over the same situations
    rng = np.random.default_rng(0)
    idx = rng.integers(0, n, (2000, n))
    paired = []
    for g in ("base", "prompted", "prompted_all", "SFT", "bandit"):
        diff = hits["DITTO"] - hits[g]
        boots = diff[idx].mean(axis=1)
        paired.append({"comparison": f"DITTO - {g}", "difference (points of agreement, %)": round(100 * diff.mean(), 1),
                       "95% CI": f"{100 * np.quantile(boots, 0.025):.1f} to {100 * np.quantile(boots, 0.975):.1f}"})
    rules = sorted({p["rule"] for p in res["base"]["per_example"]})
    rule_of = [p["rule"] for p in res["base"]["per_example"]]
    per_rule = {}
    for g in GROUPS:
        for rule in rules:
            h = np.array([x for x, r in zip(hits[g], rule_of) if r == rule])
            k = len(GROUPS[g])
            per_rule.setdefault(rule, {})[g] = fmt_agree(round(float(h.sum() * k) / k, 2), len(h), bootstrap_ci(h))
    lines = [f"## {split} ({n} situations, evaluated once)", "",
             "| model | runs | agrees end to end (mean over seeds, 95% CI) | range over seeds | valid output | median latency |",
             "|---|---|---|---|---|---|"]
    lines += [f"| {r['model']} | {r['runs']} | {r['agree']} | {r['range over seeds']} | {r['valid output']} | "
              f"{r['median latency ms']} ms |" for r in summary]
    lines += ["", "Paired differences on the same situations (DITTO averaged over seeds):", "",
              "| comparison | difference (percentage points) | 95% CI |", "|---|---|---|"]
    lines += [f"| {r['comparison']} | {r['difference (points of agreement, %)']:+.1f} | {r['95% CI']} |" for r in paired]
    lines += ["", f"Per-rule agreement on {split} (SFT and DITTO: mean over seeds per situation):", "",
              "| rule | " + " | ".join(GROUPS) + " |", "|---|" + "---|" * len(GROUPS)]
    lines += [f"| {rule} | " + " | ".join(per_rule[rule][g] for g in GROUPS) + " |" for rule in rules]
    lines += ["", f"Full table ({split}):", "", to_markdown(combined)]
    return lines, {"summary": summary, "paired": paired, "per_rule": per_rule}


dt = choice["dev_table"]
lines = [f"# DITTO on Colab: chosen variant `{CHOSEN}`", "",
         f"GPU {GPU}, dtype {DTYPE}, commit `{COMMIT[:8]}`. Variant: {VARIANTS[CHOSEN]['what']}. "
         "Training data: 119 situations (v2 = v3 training rows) generated from demonstrator A's written strategy, "
         "pending human review. prompted = the 20 v1 training demonstrations in the prompt; prompted_all = every "
         "training demonstration that fits the context window.", "",
         "## Tuning on dev (33 situations; test never read)", "",
         "| variant | base model | decoding | DITTO | SFT | base | DITTO latency | train min |", "|---|---|---|---|---|---|---|---|"]
lines += [f"| {r['variant']} | {r['base model']} | {r['decoding']} | {r['DITTO (95% CI)']} | {r['SFT (95% CI)']} | "
          f"{r['base (95% CI)']} | {r['DITTO median latency ms']:.0f} ms | {r['train min']} |" for r in dt]
lines += ["", f"Choice: {choice['why']}. Delivered seed (chosen on dev): {BEST_SEED}.", ""]
analysis = {}
for split in TEST_SPLITS:
    part, analysis[split] = analyse(split)
    lines += part + [""]
lines += ["With option scoring, every output is a valid option by construction.",
          "Mock preferences of a fictional demonstrator: an exploratory check, not a statistical or clinical validation."]
(RESULTS / "ditto_colab_test.md").write_text("\n".join(lines) + "\n")
(RESULTS / "ditto_colab_summary.json").write_text(json.dumps(
    {"chosen": CHOSEN, "why": choice["why"], "best_seed": BEST_SEED, "seed_dev": seed_dev, "analysis": analysis,
     "gpu": GPU, "dtype": DTYPE, "commit": COMMIT,
     "stage_minutes": {k: round(v / 60, 1) for k, v in STAGE_TIMES.items()}}, indent=1, default=str))
for split in TEST_SPLITS:
    print(f"\n=== {split}")
    display(pd.DataFrame(analysis[split]["summary"]))
    display(pd.DataFrame(analysis[split]["paired"]))
    display(pd.DataFrame(analysis[split]["per_rule"]).T)
print(f"report: {RESULTS / 'ditto_colab_test.md'}")

## 8. Bundle: best adapter, all runs and tables in one zip
`best_adapter/<run_id>/` has the same layout as `models/ditto_llm/<run_id>/` in the repo (final adapter,
manifest, dev, test and test_v1 evaluations).

In [ ]:
best_rid = run_id(CHOSEN, BEST_SEED)
m = manifest(best_rid)
best_dir = DRIVE / "best_adapter" / best_rid
best_dir.mkdir(parents=True, exist_ok=True)
for f in ("adapter_config.json", "adapter_model.safetensors"):
    shutil.copy(RUNS / best_rid / m["final_adapter"] / f, best_dir / f)
shutil.copy(RUNS / best_rid / "manifest.json", best_dir / "manifest.json")
for split in ("dev", "test", "test_v1"):
    shutil.copy(EVAL / f"{best_rid}-{split}.json", best_dir / f"eval_{split}.json")

import zipfile


def make_zip(name: str, parts: list[Path]) -> Path:
    # Built in /content (never inside the folder being zipped), then copied to Drive.
    tmp = (Path("/content") if Path("/content").exists() else Path(__import__("tempfile").gettempdir())) / name
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
        for part in parts:
            for p in ([part] if part.is_file() else sorted(part.rglob("*"))):
                if p.is_file():
                    z.write(p, p.relative_to(DRIVE))
    shutil.copy(tmp, DRIVE / name)
    print(f"{DRIVE / name}: {tmp.stat().st_size / 1e6:.0f} MB")
    return tmp


meta = [DRIVE / f for f in ("choice.json", "seed_choice.json", "run_info.json")]
full = make_zip("ditto_colab_bundle.zip", [RUNS, EVAL, RESULTS, CONFIGS, LOGS, DRIVE / "best_adapter", *meta])
small = make_zip("ditto_colab_best_and_results.zip", [best_dir, RESULTS, CONFIGS, *meta])
print("stage minutes:", {k: round(v / 60, 1) for k, v in STAGE_TIMES.items()})
from google.colab import files
files.download(str(small))   # best adapter + tables; the full bundle (all runs) is in Drive: ditto_colab_bundle.zip